# Predictive Analysis: Classification

**Author:** Aung Paing Soe


## Prepare

Use a stratified holdout and a preprocessing pipeline to prevent leakage.


In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
df=pd.read_csv('dataset_part_2.csv'); features=['FlightNumber','PayloadMass','Flights','GridFins','Reused','Legs','Block','ReusedCount','Orbit','LaunchSite']; X=df[features]; y=df.Class
num=X.select_dtypes(include=np.number).columns.tolist(); cat=[c for c in X if c not in num]
prep=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)


## Tune four models

Compare five-fold CV accuracy and evaluate the selected model on unseen launches.


In [ ]:
models={'Logistic Regression':(LogisticRegression(max_iter=2000),{'model__C':[.01,.1,1,10]}),'SVM':(SVC(),{'model__C':[.1,1,10],'model__kernel':['linear','rbf']}),'KNN':(KNeighborsClassifier(),{'model__n_neighbors':[3,5,7,9]}),'Decision Tree':(DecisionTreeClassifier(random_state=42),{'model__max_depth':[2,3,4,5,6],'model__min_samples_leaf':[1,2,4]})}
results=[]; fitted={}
for name,(model,grid) in models.items():
    search=GridSearchCV(Pipeline([('prep',prep),('model',model)]),grid,cv=5,scoring='accuracy'); search.fit(X_train,y_train); pred=search.predict(X_test)
    results.append([name,search.best_score_,accuracy_score(y_test,pred)]); fitted[name]=(search,pred)
display(pd.DataFrame(results,columns=['Model','CV Accuracy','Test Accuracy']).sort_values('CV Accuracy',ascending=False))


## Error analysis

Interpret the confusion matrix as TN, FP, FN, TP. False positives can understate expected recovery cost in a bid.


In [ ]:
best=max(results,key=lambda x:x[1])[0]; search,pred=fitted[best]
print(best,search.best_params_); print(confusion_matrix(y_test,pred)); print(classification_report(y_test,pred))
